# Ulrich et al. 2024 atlas: decidual stroma vs C7-Fibroblast (PyDESeq2)

The decidual-versus-fibroblast comparison of the X-Mens and Slide-tags fits, repeated in an independent human
atlas: the full-thickness uterus single-cell RNA-seq atlas of Ulrich et al. 2024 (PNAS 121:e2404775121; 20 donors),
in its CELLxGENE stromal subset. Decidual stroma (author labels dS1, dS2 and dS3, pooled) is compared with
C7-Fibroblasts within donor: each donor with at least 20 cells of both gives one pseudobulk per state, and the donor
term absorbs the differences between donors. Ten donors qualify, among them D3, whose menstrual phase is not
recorded.

- **Design** `~ donor + state`: 20 pseudobulks (10 donors × 2 states), 9 residual degrees of freedom.
- **Test** PyDESeq2 Wald test of decidual vs reference (C7-Fibroblast); a positive log2 fold change is higher in
  decidual stroma. Genes tested: at least 1 CPM in at least half of the pseudobulks of either state. Adjusted
  p-values are Benjamini-Hochberg over the tested genes, without independent filtering. DE: adjusted p < 0.05 and
  |unshrunk log2 fold change| > 1.
- **Record** contrast `dS_vs_C7fib_all_donors` of release v7 of `ulrich_ivd_pipeline.py` (28 October 2026): 13,772
  genes tested, 886 DE (337 up, 549 down). The release's config, `config_ulrich_dS_vs_C7fib_luteal_follicular.yaml`,
  is named after another of its contrasts.
- **Inputs** the CELLxGENE object (counts in `raw/X`) and the feature manifest of record.
- **Outputs** in `OUT_DIR`: the all-genes table, the pseudobulk table and the pseudobulk counts, with the file names
  and formats of the release.
- **Runtime** about 2 minutes on 8 CPUs, with a peak of about 4 GB of memory.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them. Relative paths start from this notebook's folder.

# Inputs of record. The h5ad is the unmodified CELLxGENE download; see DOWNLOAD_H5AD.
H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/ulrich2024/ulrich2024_20donor_stromal.h5ad"
FEATURE_MANIFEST = "resources/ulrich_feature_manifest.tsv"
DOWNLOAD_H5AD = False  # True downloads the atlas from CELLxGENE into H5AD if H5AD does not exist

# The tables of record (release v7, 28 October 2026), compared with this run at the end; None skips that comparison.
RECORD = "record/ulrich_dS_vs_C7fib_all_donors_all_genes.csv.gz"
RECORD_PSEUDOBULKS = "record/ulrich_dS_vs_C7fib_all_donors_metadata.tsv"

OUT_DIR = None  # None writes nothing
FORCE = False  # True lets this run replace the outputs already in OUT_DIR
N_CPUS = 8  # PyDESeq2 workers

In [ ]:
import hashlib
import io
import platform
import shutil
import urllib.request
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pydeseq2
import scipy
from anndata.io import read_elem, sparse_dataset
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

print(
    f"Python {platform.python_version()}, PyDESeq2 {pydeseq2.__version__}, NumPy {np.__version__}, "
    f"SciPy {scipy.__version__}, pandas {pd.__version__}"
)

## Settings

The settings of the fit of record, from the release's config.

In [ ]:
CONTRAST = "dS_vs_C7fib_all_donors"  # the contrast's id in the release; names the output files

# author_cell_type labels pooled into each state; "decidual" and "reference" are the release's state names.
STATE_LABELS = {
    "decidual": ["dS1", "dS2", "dS3"],
    "reference": ["C7-Fibroblast"],
}

MIN_CELLS = 20  # per pseudobulk
MIN_CPM = 1.0  # gene eligibility; see Pseudobulks
MAX_PADJ = 0.05  # DE: adjusted p below this ...
MIN_ABS_LOG2FC = 1.0  # ... and |unshrunk log2 fold change| above this

# CELLxGENE dataset c93622aa-a356-4e29-b45a-c0a532c65d10: 85,875 cells x 36,166 genes, 1,283,567,526 bytes.
H5AD_URL = "https://datasets.cellxgene.cziscience.com/c93622aa-a356-4e29-b45a-c0a532c65d10.h5ad"
H5AD_SHA256 = "5be96393e3a18e9596eeb67e8c0a024af54e78effd2c5698cb92876961734a90"

In [ ]:
def sha256_of(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(16 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


# The release wrote its tables with run_pipeline.py's write_csv and write_tsv: 15 significant digits, missing values
# empty in CSV and NA in TSV. Writing the same way makes the outputs comparable with it byte for byte.
def release_csv(table: pd.DataFrame) -> str:
    return table.to_csv(index=False, na_rep="", float_format="%.15g", lineterminator="\n")


def release_tsv(table: pd.DataFrame) -> str:
    return table.to_csv(sep="\t", index=False, na_rep="NA", float_format="%.15g", lineterminator="\n")

## Load

The atlas is the unmodified CELLxGENE download. Its copy of record is on netscratch, which FASRC purges;
`DOWNLOAD_H5AD = True` fetches it again into `H5AD` (1.3 GB). Either way the file must have the SHA-256 that the fit
of record checked.

In [ ]:
# The only cell that writes outside OUT_DIR, and only when asked to. A download that fails the checksum is left
# as H5AD.partial and not used.
if DOWNLOAD_H5AD and not Path(H5AD).exists():
    partial = Path(f"{H5AD}.partial")
    partial.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(H5AD_URL) as response, partial.open("wb") as handle:
        shutil.copyfileobj(response, handle, 16 << 20)
    if sha256_of(partial) != H5AD_SHA256:
        raise ValueError(f"{partial} does not have the SHA-256 of record")
    partial.rename(H5AD)
    print(f"downloaded {H5AD}")

In [ ]:
if sha256_of(H5AD) != H5AD_SHA256:
    raise ValueError(f"{H5AD} is not the atlas of record (SHA-256 differs)")

# Counts come from raw/X, the raw UMI counts of the CELLxGENE object, as in the fit of record.
with h5py.File(H5AD, "r") as handle:
    obs = read_elem(handle["obs"])
    genes = read_elem(handle["raw/var"]).index.astype(str)

# The manifest of record maps the features of raw/X one to one onto genes (none excluded, none merged), so it adds
# only the gene_symbol and biotype columns of the table of record. Stop if that no longer holds.
manifest = pd.read_csv(FEATURE_MANIFEST, sep="\t")
is_one_to_one = (
    manifest["source_index"].tolist() == list(range(len(genes)))
    and manifest["source_feature_id"].astype(str).tolist() == genes.tolist()
    and manifest["canonical_id"].astype(str).tolist() == genes.tolist()
    and not manifest["excluded"].any()
)
if not is_one_to_one:
    raise ValueError(f"{FEATURE_MANIFEST} does not map raw/var one to one")

print(f"{len(obs):,} cells x {len(genes):,} genes")

## Pseudobulks

Cells labeled dS1, dS2 or dS3 are decidual stroma and cells labeled C7-Fibroblast the reference; the other
stromal populations are left out. Counts are summed per donor and state. A donor is kept only if both of its
pseudobulks have at least 20 cells, so that every decidual pseudobulk is compared with C7-Fibroblasts of the same
donor.

In [ ]:
label_to_state = {label: state for state, labels in STATE_LABELS.items() for label in labels}

cells = pd.DataFrame(
    {
        "row": np.arange(len(obs)),  # position in raw/X
        "donor": obs["donor_id"].astype(str),
        "label": obs["author_cell_type"].astype(str),
        "tissue": obs["tissue"].astype(str),
        "menstrual_phase_at_collection": obs["menstrual_phase_at_collection"].astype(str),
        "Data_source": obs["Data_source"].astype(str),
    }
)
cells["state"] = cells["label"].map(label_to_state)
cells = cells[cells["state"].notna()].reset_index(drop=True)

# Phase and data source describe donors: one value per donor, which the pseudobulk table carries.
values_per_donor = cells.groupby("donor")[["menstrual_phase_at_collection", "Data_source"]].nunique()
if values_per_donor.gt(1).any().any():
    raise ValueError("menstrual phase or data source varies within a donor")

print(f"{len(cells):,} cells in the two states")
pd.crosstab(cells["donor"], cells["label"])

In [ ]:
# raw/X stores the UMI counts as float32; they are summed as int64, as in the fit of record, so every sum is exact.
with h5py.File(H5AD, "r") as handle:
    cell_counts = sparse_dataset(handle["raw/X"])[cells["row"].to_numpy()]

if not np.array_equal(cell_counts.data, np.round(cell_counts.data)) or cell_counts.data.min() < 0:
    raise ValueError("raw/X does not hold non-negative whole-number counts")
cell_counts = cell_counts.astype(np.int64)

# One row per donor and state, sorted by donor (as text) and then state, the order of the fit of record. Row order
# matters: PyDESeq2 sums over samples in this order, and the last digits of its estimates depend on it.
rows_by_pseudobulk = cells.groupby(["donor", "state"]).indices
counts_all = pd.DataFrame(
    [np.asarray(cell_counts[rows].sum(axis=0)).ravel() for rows in rows_by_pseudobulk.values()],
    index=pd.MultiIndex.from_tuples(list(rows_by_pseudobulk), names=["donor", "state"]),
    columns=genes,
).sort_index()

print(f"{len(counts_all)} donor-state pseudobulks before the cell floor")

In [ ]:
def label_counts(labels: pd.Series) -> str:
    # e.g. "dS1:108;dS2:2423;dS3:1445", labels in sorted order
    return ";".join(f"{label}:{n}" for label, n in labels.value_counts().sort_index().items())


by_pseudobulk = cells.groupby(["donor", "state"])
pseudobulks = by_pseudobulk.agg(
    n_cells=("label", "size"),
    labels=("label", label_counts),
    menstrual_phase_at_collection=("menstrual_phase_at_collection", "first"),
    Data_source=("Data_source", "first"),
)
tissue_counts = pd.crosstab([cells["donor"], cells["state"]], cells["tissue"]).add_prefix("n_cells_")
pseudobulks = pseudobulks.join(tissue_counts)

# Every donor of the atlas has more than 20 decidual stroma cells; the C7-Fibroblasts decide which donors stay.
cells_per_state = pseudobulks["n_cells"].unstack("state", fill_value=0)
is_kept = cells_per_state.ge(MIN_CELLS).all(axis=1)
dropped = cells_per_state[~is_kept]
dropped_text = ", ".join(
    f"{donor} ({n_decidual} / {n_reference})"
    for donor, n_decidual, n_reference in zip(dropped.index, dropped["decidual"], dropped["reference"])
)
print(
    f"Kept {is_kept.sum()} of {len(is_kept)} donors. Dropped, with fewer than {MIN_CELLS} cells in a state "
    f"(decidual / C7-Fibroblast cells): {dropped_text}"
)

pseudobulks = pseudobulks.loc[is_kept.index[is_kept]]
counts = counts_all.loc[pseudobulks.index]

pseudobulks = pseudobulks.reset_index()
pseudobulks.insert(0, "pseudobulk_id", "donor=" + pseudobulks["donor"] + "__state=" + pseudobulks["state"])
pseudobulks["library_size"] = counts.sum(axis=1).to_numpy()
counts.index = pd.Index(pseudobulks["pseudobulk_id"], name="pseudobulk_id")

cells_kept = pseudobulks.groupby("state")["n_cells"].sum()
print(
    f"{len(pseudobulks)} pseudobulks, {cells_kept.sum():,} cells: {cells_kept['decidual']:,} decidual stroma, "
    f"{cells_kept['reference']:,} C7-Fibroblasts"
)

# One row per kept donor: cells and summed counts per state
(
    pseudobulks.groupby("donor")[["menstrual_phase_at_collection", "Data_source"]]
    .first()
    .join(pseudobulks.pivot(index="donor", columns="state", values="n_cells").add_prefix("cells_"))
    .join(pseudobulks.pivot(index="donor", columns="state", values="library_size").add_prefix("counts_"))
)

A gene is tested if it has at least 1 CPM in at least half of the pseudobulks of either state, so that genes
expressed in only one of the two states are kept. This rule is the only gene filter.

In [ ]:
# CPM over all 36,166 genes. "Half" is (n + 1) // 2 pseudobulks: 5 of the 10 of each state.
cpm = counts.div(counts.sum(axis=1), axis=0) * 1e6
is_expressed = cpm >= MIN_CPM

is_eligible = pd.Series(False, index=genes)
for state in STATE_LABELS:
    in_state = (pseudobulks["state"] == state).to_numpy()
    is_eligible |= is_expressed[in_state].sum(axis=0) >= (in_state.sum() + 1) // 2

print(f"{is_eligible.sum():,} of {len(genes):,} genes eligible")

## Fit

PyDESeq2 with the settings of record: design `~ donor + state` with Cook's outlier refitting on, then the Wald test
of decidual vs reference without independent filtering, and apeGLM shrinkage for the shrunk fold changes.

In [ ]:
# Donor and state are text, so their reference levels are the first in sort order: donor 39 and state "decidual".
metadata = pseudobulks.set_index("pseudobulk_id")[["donor", "state"]]
inference = DefaultInference(n_cpus=N_CPUS)

# refit_cooks=True is the setting of record, but it replaces nothing here: PyDESeq2 refits outliers only in samples
# with at least 7 replicates (min_replicates), and each pseudobulk is the only one of its donor and state.
dds = DeseqDataSet(
    counts=counts.loc[:, is_eligible.to_numpy()],
    metadata=metadata,
    design="~ donor + state",
    refit_cooks=True,
    inference=inference,
    quiet=True,
)
dds.deseq2()

design = dds.obsm["design_matrix"]
residual_df = len(design) - np.linalg.matrix_rank(design.to_numpy(dtype=float))
print(f"{dds.n_obs} pseudobulks x {dds.n_vars:,} genes")
print(f"design: {design.shape[1]} columns, {residual_df} residual degrees of freedom")
print(f"size factors {dds.obs['size_factors'].min():.3f} to {dds.obs['size_factors'].max():.2f}")
print(f"genes with counts replaced as outliers: {int(dds.var['replaced'].sum())}")

In [ ]:
# independent_filter=False: the eligibility rule is the only gene filter, so padj is Benjamini-Hochberg over exactly
# the tested genes. Cook's filtering of p-values (on by default) looks only at samples with at least 3 replicates,
# so it removes no p-value here either.
stats = DeseqStats(
    dds,
    contrast=["state", "decidual", "reference"],
    independent_filter=False,
    inference=inference,
    quiet=True,
)
stats.summary()
wald = stats.results_df.copy()  # lfc_shrink overwrites results_df

# apeGLM shrinks a coefficient of the design, and the design's state coefficient, state[T.reference], is reference
# over decidual. Its sign is flipped so that the shrunk fold change reads decidual over reference like the others.
stats.lfc_shrink(coeff="state[T.reference]")
lfc_shrunk = -stats.results_df["log2FoldChange"]

print(f"p-values missing: {int(wald['pvalue'].isna().sum())}")

## Results

One row per gene of the atlas (36,166), with statistics for the tested genes, in the columns of the release, so that
the workbook scripts read this table as they read the release's.

In [ ]:
results = pd.DataFrame(
    {
        "ensembl_gene_id": genes,
        "gene_symbol": manifest["gene_symbol"].to_numpy(),
        "biotype": manifest["biotype"].to_numpy(),
        "source_feature_ids": genes,
        "source_feature_n": 1,
        "carrier": True,
        "production_eligible": is_eligible.to_numpy(),
        "total_count": counts.sum(axis=0).to_numpy(),
        "nonzero_pseudobulks": (counts > 0).sum(axis=0).to_numpy(),
    }
)

statistics = pd.DataFrame(
    {
        "baseMean": wald["baseMean"],
        "log2FoldChange_unshrunk": wald["log2FoldChange"],
        "log2FoldChange_shrunk": lfc_shrunk,
        "lfcSE": wald["lfcSE"],
        "stat": wald["stat"],
        "pvalue": wald["pvalue"],
        "padj": wald["padj"],
    }
)
results = results.join(statistics, on="ensembl_gene_id")

# A tested gene without a finite fold change or p-value would be reported as not estimable, without statistics;
# every tested gene is estimable here.
results["estimable"] = (
    results["production_eligible"]
    & np.isfinite(results["log2FoldChange_unshrunk"])
    & np.isfinite(results["pvalue"])
)
stat_columns = ["log2FoldChange_unshrunk", "log2FoldChange_shrunk", "lfcSE", "stat", "pvalue", "padj"]
results.loc[~results["estimable"], stat_columns] = np.nan

# Untested genes count as padj 1 in the DE rule.
results["padj_for_call"] = results["padj"].where(np.isfinite(results["padj"]), 1.0)
results["significant_fdr05_lfc1"] = (
    results["estimable"]
    & (results["padj_for_call"] < MAX_PADJ)
    & (results["log2FoldChange_unshrunk"].abs() > MIN_ABS_LOG2FC)
)

# Constant columns of the release's table
n_decidual = int((pseudobulks["state"] == "decidual").sum())
results = results.assign(
    source_id=f"ulrich_{CONTRAST}",
    positive_lfc_meaning="decidual_minus_reference",
    analysis_method="pydeseq2_wald_fixed_block",
    n_pseudobulks_numerator=n_decidual,
    n_pseudobulks_denominator=len(pseudobulks) - n_decidual,
)

In [ ]:
is_de = results["significant_fdr05_lfc1"]
n_up = int((is_de & (results["log2FoldChange_unshrunk"] > 0)).sum())
n_down = int((is_de & (results["log2FoldChange_unshrunk"] < 0)).sum())
n_tested = int(results["production_eligible"].sum())
print(f"{n_tested:,} genes tested, {int(is_de.sum())} DE: {n_up} up, {n_down} down")

shown = ["gene_symbol", "baseMean", "log2FoldChange_unshrunk", "log2FoldChange_shrunk", "padj"]
results.loc[is_de].set_index("ensembl_gene_id")[shown].sort_values("padj").head(10)

## Check against the record

The tables are compared as written, to the 15 significant digits the release kept: the same genes, and every column
identical (NaN equal to NaN), which includes the DE calls. The pseudobulk table is compared as text. A difference
stops the notebook before anything is saved.

In [ ]:
def compare_columns(ours: pd.DataFrame, record: pd.DataFrame, key: str) -> pd.DataFrame:
    # Per column present in both: values that differ (NaN equal to NaN) and, for numbers, the largest difference.
    ours = ours.set_index(key)
    record = record.set_index(key).loc[ours.index]
    rows = []
    for column in ours.columns.intersection(record.columns):
        mine, theirs = ours[column], record[column]
        differs = (mine != theirs) & ~(mine.isna() & theirs.isna())
        is_number = pd.api.types.is_float_dtype(mine) or pd.api.types.is_integer_dtype(mine)
        rows.append(
            {
                "column": column,
                "n_differing": int(differs.sum()),
                "max_abs_difference": float((mine - theirs).abs().max()) if is_number else np.nan,
            }
        )
    return pd.DataFrame(rows)


if RECORD is None:
    gene_check = "skipped"
else:
    ours = pd.read_csv(io.StringIO(release_csv(results)), float_precision="round_trip")
    record = pd.read_csv(RECORD, float_precision="round_trip")

    only_ours = set(ours["ensembl_gene_id"]) - set(record["ensembl_gene_id"])
    only_record = set(record["ensembl_gene_id"]) - set(ours["ensembl_gene_id"])
    if only_ours or only_record:
        raise ValueError(f"genes differ: {len(only_ours)} only here, {len(only_record)} only in the record")

    comparison = compare_columns(ours, record, "ensembl_gene_id")
    print(comparison.to_string(index=False))

    missing_columns = record.columns.difference(ours.columns).tolist()
    if comparison["n_differing"].any() or missing_columns:
        raise ValueError(f"results differ from the record (columns missing here: {missing_columns})")
    n_de_record = int(record["significant_fdr05_lfc1"].sum())
    gene_check = f"identical ({len(record):,} genes, {len(record.columns)} columns, {n_de_record} DE)"

print(f"all-genes table: {gene_check}")

In [ ]:
if RECORD_PSEUDOBULKS is None:
    pseudobulk_check = "skipped"
else:
    ours_text = pd.read_csv(io.StringIO(release_tsv(pseudobulks)), sep="\t", dtype=str, keep_default_na=False)
    record_text = pd.read_csv(RECORD_PSEUDOBULKS, sep="\t", dtype=str, keep_default_na=False)
    if list(ours_text.columns) != list(record_text.columns) or len(ours_text) != len(record_text):
        raise ValueError(f"pseudobulk tables differ: {ours_text.shape} here, {record_text.shape} in the record")

    differing = [column for column in ours_text.columns if not ours_text[column].equals(record_text[column])]
    if differing:
        print(ours_text.compare(record_text).to_string())
        raise ValueError(f"pseudobulks differ from the record in {differing}")
    pseudobulk_check = f"identical ({len(record_text)} pseudobulks, {len(record_text.columns)} columns)"

print(f"pseudobulks: {pseudobulk_check}")

## Save

Writes only into `OUT_DIR`, which is created here, after the check: a failed run leaves no outputs behind.

In [ ]:
if OUT_DIR is None:
    print("OUT_DIR is None; nothing written")
else:
    out_dir = Path(OUT_DIR)
    outputs = {
        f"ulrich_{CONTRAST}_all_genes.csv": release_csv(results),
        f"ulrich_{CONTRAST}_metadata.tsv": release_tsv(pseudobulks),
        f"ulrich_{CONTRAST}_counts.tsv": release_tsv(counts.reset_index()),
    }
    existing = [name for name in outputs if (out_dir / name).exists()]
    if existing and not FORCE:
        raise FileExistsError(f"{out_dir / existing[0]} exists; set FORCE = True to overwrite")

    out_dir.mkdir(parents=True, exist_ok=True)
    for name, text in outputs.items():
        (out_dir / name).write_text(text)
        print(f"wrote {out_dir / name}")

## Summary

In [ ]:
n_donors = pseudobulks["donor"].nunique()
print(f"pseudobulks: {len(pseudobulks)} ({n_donors} donors, {pseudobulks['n_cells'].sum():,} cells)")
print(f"genes tested: {n_tested:,} of {len(results):,}")
print(f"DE (padj < {MAX_PADJ}, |log2 FC| > {MIN_ABS_LOG2FC:g}): {int(is_de.sum())} ({n_up} up, {n_down} down)")
print(f"residual degrees of freedom: {residual_df}")
print(f"check against the record: all-genes table {gene_check}; pseudobulks {pseudobulk_check}")